In [ ]:
!pip install transformers scikit-learn faiss-gpu datasets huggingface_hub bitsandbytes

This is an implementation of a Standard RAG pipeline for a biomedical Q&A application focused on diseases. It compares sparse retrieval (using TF-IDF) and dense retrieval (using Dense Passage Retrieval, DPR) while using Llama 2 as the generative model.

Data Preparation

We'll use the CORD-19 dataset or PubMed articles (freely available) for biomedical information. This dataset contains scientific papers and abstracts related to biomedicine and diseases.

In [ ]:
from datasets import load_dataset

# Load a biomedical dataset (e.g., Cord19 subset from PubMed dataset)
dataset = load_dataset("allenai/cord19", name ="metadata", split="train", trust_remote_code=True)

# Shuffle the dataset with a fixed seed
shuffled_dataset = dataset.shuffle(seed=42)

# Select the first 10% of the shuffled dataset
subset_size = int(0.1 * len(shuffled_dataset))  # Calculate 10% of the dataset size
subset = shuffled_dataset.select(range(subset_size))

# Preprocess the dataset for retrieval ilter the dataset for specific topics
documents = [doc for doc in subset if 'abstract' in doc and doc['abstract'] and 'diabetes' in doc['abstract'].lower()]

# Titles for better visualization
titles = [doc['title'] for doc in documents if 'title' in doc and doc['title']]

print(f"Loaded {len(documents)} documents related to diabetes.")
print(f"Sample titles: {titles[:5]}")


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

cord19.py:   0%|          | 0.00/8.42k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/8.13k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/368618 [00:00<?, ? examples/s]

Loaded 627 documents related to diabetes.
Sample titles: ['Insulin treatment is associated with increased mortality in patients with COVID-19', 'Comorbidity and its impact on 1590 patients with COVID-19 in China: a nationwide analysis', 'Sociodemographic, clinical and laboratory factors on admission associated with COVID-19 mortality in hospitalized patients: A retrospective observational study', 'The Value of Telemedicine for the Follow-up of Patients with New Onset Type 1 Diabetes Mellitus During COVID-19 Pandemic in Turkey: A Report of Eight Cases', 'Cardiovascular comorbidities as predictors for severe COVID-19 infection or death']


Sparse Retrieval (TF-IDF)

Use TF-IDF to vectorize documents and retrieve relevant text.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform([doc['abstract'] for doc in documents])

def sparse_retrieval(query, top_k=5):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_indices = scores.argsort()[-top_k:][::-1]
    return [(titles[i], documents[i]) for i in top_indices]

# Example query
query = "What are the symptoms of diabetes?"
results = sparse_retrieval(query)
print("Top results from TF-IDF:")
for title, doc in results:
    print(f"Title: {title}\nDocument: {doc}\n")


Top results from TF-IDF:
Title: Clinical characteristics, predictors of symptomatic coronavirus disease 2019 and duration of hospitalisation in a cohort of 632 Patients in Lagos State, Nigeria.
Document: {'cord_uid': '56hws0oz', 'sha': '', 'source_x': 'Medline', 'title': 'Clinical characteristics, predictors of symptomatic coronavirus disease 2019 and duration of hospitalisation in a cohort of 632 Patients in Lagos State, Nigeria.', 'doi': '10.4103/npmj.npmj_272_20', 'abstract': 'Objective The clinical spectrum of severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) infection is still evolving. This study describes the clinical characteristics and investigates factors that predict symptomatic presentation and duration of hospitalisation in a cohort of coronavirus disease 2019 (COVID-19) patients managed in Lagos, Nigeria. Methodology This was a retrospective assessment of patients hospitalised with COVID-19 disease in six dedicated facilities in Lagos, Nigeria, between April 1s

Dense Passage Retrieval (DPR)

Use Hugging Face's Dense Passage Retrieval (DPR) to embed and retrieve passages.

In [ ]:
from transformers import DPRContextEncoder, DPRQuestionEncoder, DPRContextEncoderTokenizer, DPRQuestionEncoderTokenizer
import numpy as np
import faiss

# Load DPR models and tokenizers
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

# Encode documents for dense retrieval
context_embeddings = []
for doc in documents:
    inputs = context_tokenizer(doc['abstract'], return_tensors="pt", truncation=True, max_length=512)
    embedding = context_encoder(**inputs).pooler_output.detach().numpy()
    context_embeddings.append(embedding)

context_embeddings = np.vstack(context_embeddings)

# Use FAISS for efficient similarity search
faiss_index = faiss.IndexFlatIP(context_embeddings.shape[1])
faiss_index.add(context_embeddings)

def dense_retrieval(query, top_k=5):
    query_inputs = question_tokenizer(query, return_tensors="pt", truncation=True, max_length=512)
    query_embedding = question_encoder(**query_inputs).pooler_output.detach().numpy()
    scores, indices = faiss_index.search(query_embedding, top_k)
    return [(titles[i], documents[i]) for i in indices[0]]

# Example query
results = dense_retrieval(query)
print("Top results from DPR:")
for title, doc in results:
    print(f"Title: {title}\nDocument: {doc}\n")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Top results from DPR:
Title: Epigenetic Effects of Gut Metabolites: Exploring the Path of Dietary Prevention of Type 1 Diabetes
Document: {'cord_uid': 'ngizkcjd', 'sha': '80f2f7b87e40f29c441db5b7ae973616a2a2871f', 'source_x': 'PMC', 'title': 'Epigenetic Effects of Gut Metabolites: Exploring the Path of Dietary Prevention of Type 1 Diabetes', 'doi': '10.3389/fnut.2020.563605', 'abstract': 'Type 1 diabetes (T1D) has increased over the past half century and has now become the second most frequent autoimmune disease in childhood and one of major public health concern worldwide. Evidence suggests that modern lifestyles and rapid environmental changes are driving factors that underlie this increase. The integration of these two factors brings about changes in food intake. This, in turn, alters epigenetic regulations of the genome and intestinal microbiota composition, which may ultimately play a role in pathogenesis of T1D. Recent evidence shows that dysbiosis of the gut microbiota is closel

Generation Using LLaMA 2

Use LLaMA 2 (through Hugging Face Transformers) to synthesize answers from retrieved documents.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve and log in with the token directly
login(userdata.get("HF_TOKEN"))

# New Section

In [ ]:
import torch
import bitsandbytes as bnb
from transformers import  AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

print("Bitsandbytes version:", bnb.__version__)
print("GPU available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

quantization_config = BitsAndBytesConfig(load_in_8bit=True,llm_int8_enable_fp32_cpu_offload=True)

Bitsandbytes version: 0.45.0
GPU available: True
CUDA version: 12.1


In [ ]:
from tqdm import tqdm

# Load LLaMA 2 model and tokenizer
model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token = True)
llm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, # datatype to use, we want float16
                                                 quantization_config=quantization_config)

def generate_answer(query, retrieved_docs):
    # Concatenate retrieved documents and the query
    context = "\n\n".join([doc['abstract'] for _, doc in retrieved_docs if 'abstract' in doc])
    input_text = f"Query: {query}\n\nContext: {context}\n\nAnswer:"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024)
    outputs = llm.generate(**inputs, max_length=2048, temperature=0.7, top_p=0.9, repetition_penalty=1.1)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Generate an answer
retrieved_docs = dense_retrieval(query)  # Switch between sparse_retrieval and dense_retrieval
answer = generate_answer(query, retrieved_docs)
print("Generated Answer:")
print(answer)


/usr/local/lib/python3.10/dist-packages/transformers/models/auto/tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:2097: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Generated Answer:
Query: What are the symptoms of diabetes?

Context: Type 1 diabetes (T1D) has increased over the past half century and has now become the second most frequent autoimmune disease in childhood and one of major public health concern worldwide. Evidence suggests that modern lifestyles and rapid environmental changes are driving factors that underlie this increase. The integration of these two factors brings about changes in food intake. This, in turn, alters epigenetic regulations of the genome and intestinal microbiota composition, which may ultimately play a role in pathogenesis of T1D. Recent evidence shows that dysbiosis of the gut microbiota is closely associated with T1D and that a dietary intervention can influence epigenetic changes associated with this disease and may modify gene expression patterns through epigenetic mechanisms. In this review focus on how a diet can shape the gut microbiome, its effect on the epigenome in T1D, and the future of T1D management b

Evaluation and Comparison

Evaluate the performance of sparse (TF-IDF) and dense (DPR) retrieval methods using metrics like precision, recall, and ROUGE scores. This ensures the pipeline provides accurate and relevant answers.


In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=a1e4917221446912c5a0c28ed2302200dcfdddb2a70e6d52d4eceb9cd8d18404
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [ ]:

from rouge_score import rouge_scorer

def evaluate_retrieval(query, gold_answer, retriever_func, top_k=5):
    retrieved_docs = retriever_func(query, top_k=top_k)
    retrieved_text = " ".join([doc['abstract'] for _, doc in retrieved_docs if 'abstract' in doc])
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores = scorer.score(gold_answer, retrieved_text)
    return scores

# Example gold answer (for evaluation purposes)
gold_answer = "Diabetes symptoms include increased thirst, frequent urination, and fatigue."

# Evaluate TF-IDF
tfidf_scores = evaluate_retrieval(query, gold_answer, sparse_retrieval)
print("TF-IDF Scores:", tfidf_scores)

# Evaluate DPR
dpr_scores = evaluate_retrieval(query, gold_answer, dense_retrieval)
print("DPR Scores:", dpr_scores)


TF-IDF Scores: {'rouge1': Score(precision=0.004398826979472141, recall=0.6666666666666666, fmeasure=0.008739985433357612), 'rougeL': Score(precision=0.004398826979472141, recall=0.6666666666666666, fmeasure=0.008739985433357612)}
DPR Scores: {'rouge1': Score(precision=0.00916030534351145, recall=0.6666666666666666, fmeasure=0.01807228915662651), 'rougeL': Score(precision=0.007633587786259542, recall=0.5555555555555556, fmeasure=0.01506024096385542)}
